# Análisis del Motor de Búsqueda Semántica

En este notebook vamos a analizar los resultados de las búsquedas y visualizarlos.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.database import ChromaDBManager
from src.search_engine import SemanticSearchEngine

# Inicializar
db_manager = ChromaDBManager()
search_engine = SemanticSearchEngine(db_manager)

## 1. Tabla Comparativa de Resultados

In [ ]:
queries_prueba = [
    "¿cómo hacer una API en Python?",
    "diferencias entre frameworks de frontend",
    "cómo funciona la autenticación en aplicaciones web",
    "herramientas para trabajar con modelos de lenguaje"
]

data = []
for q in queries_prueba:
    results = search_engine.search(q, n_resultados=1)
    if results:
        best_result = results[0]
        data.append({
            "Query": q,
            "Mejor Resultado (Título)": best_result.article.titulo,
            "Score Similitud": round(best_result.similarity_score, 4)
        })

df = pd.DataFrame(data)
display(df)

## 2. Mapa de Calor de Similitud (Bonus)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

docs = db_manager.get_all_documents()
embeddings = docs["embeddings"]
titulos = [m["titulo"] for m in docs["metadatas"]]

if embeddings:
    # Calcular matriz de similitud del coseno
    sim_matrix = cosine_similarity(embeddings)
    
    # Plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(sim_matrix, xticklabels=titulos, yticklabels=titulos, annot=True, cmap="YlGnBu", fmt=".2f")
    plt.title("Mapa de Calor de Similitud entre Artículos")
    plt.tight_layout()
    plt.show()
else:
    print("No hay documentos indexados para generar el mapa de calor.")